In [ ]:
import networkx as nx
from pathlib import Path
from dash import Dash, html
import dash_cytoscape as cyto

NODE_TYPE_ATTR = "Node_Type"
node_type_colors = {
    "concept": "purple",
    "people": "blue",
    "location": "green",
    "conjunction": "yellow",
    "relationship": "orange",
    "target": "red",
    "other": "gray",
}

# Paths for Mac
# file1 = "/Users/nickpellegri/Documents/Github/w0rldview/test1.graphml"
# file2 = "/Users/nickpellegri/Documents/Github/w0rldview/test2.graphml"

# Paths for Windows
file1 = Path(r"C:\WorldView\worldview\w0rldview\test1.graphml")
file2 = Path(r"C:\WorldView\worldview\w0rldview\test2.graphml")

def load_and_make_elements(path, seed=42):
    G = nx.read_graphml(path)
    pos = nx.spring_layout(G, seed=seed)
    # scale/translate positions for Cytoscape (pixels)
    scale = 700
    offset = 350
    def to_pos(p):
        return {"x": float(p[0] * scale + offset), "y": float(p[1] * scale + offset)}

    nodes = []
    for n in G.nodes():
        # determine node color by type (store as data, not inline style)
        t = str(G.nodes[n].get(NODE_TYPE_ATTR, "other")).strip().lower()
        nodes.append({
            "data": {"id": str(n), "label": str(n), "type": t},    # <-- add type here
            "position": to_pos(pos[n]),
            # remove "style" so stylesheet wins
        })
        
    edges = []
    for u, v in G.edges():
        edges.append({
            "data": {"id": f"{u}__{v}", "source": str(u), "target": str(v)}
        })

    return G, nodes, edges

# Load both graphs
G1, nodes1, edges1 = load_and_make_elements(file1, seed=42)
G2, nodes2, edges2 = load_and_make_elements(file2, seed=7)  # different seed for variation

# Determine matching vs unique
nset1 = set(G1.nodes())
nset2 = set(G2.nodes())
eset1 = set(G1.edges())
eset2 = set(G2.edges())

matching_nodes = nset1 & nset2
matching_edges = eset1 & eset2

def mark_elements(nodes, edges, matching_nodes, matching_edges):
    out = []
    for n in nodes:
        n_id = n["data"]["id"]
        cls = "match" if n_id in matching_nodes else "unique"
        n["classes"] = cls
        out.append(n)
    for e in edges:
        s = e["data"]["source"]
        t = e["data"]["target"]
        cls = "match" if ( (s, t) in matching_edges or (s, t) in {(b,a) for (a,b) in matching_edges} ) else "unique"
        e["classes"] = cls
        # add arrow shape on edges
        e["data"]["directed"] = True
        out.append(e)
    return out

elements1 = mark_elements(nodes1, edges1, matching_nodes, matching_edges)
elements2 = mark_elements(nodes2, edges2, matching_nodes, matching_edges)

# Cytoscape stylesheet

stylesheet = [
    {"selector": "node", "style": {"label": "data(label)", "width": 60, "height": 60, "font-size": 10,
                                  "text-valign": "center", "text-halign": "center",
                                  "border-width": 0, "border-color": "#222"}},
    {"selector": "edge", "style": {"width": 3, "line-color": "#bbb", "target-arrow-color": "#bbb",
                                   "target-arrow-shape": "triangle", "curve-style": "bezier"}},

    # node type colors + shapes (use data selector)
    {"selector": 'node[type = "concept"]', "style": {"background-color": "purple", "shape": "rhomboid"}},
    {"selector": 'node[type = "people"]', "style": {"background-color": "blue", "shape": "ellipse"}},
    {"selector": 'node[type = "location"]', "style": {"background-color": "green", "shape": "rectangle"}},
    {"selector": 'node[type = "conjunction"]', "style": {"background-color": "yellow", "shape": "triangle"}},
    {"selector": 'node[type = "relationship"]', "style": {"background-color": "orange", "shape": "circle"}},
    {"selector": 'node[type = "target"]', "style": {"background-color": "red", "shape": "star"}},
    {"selector": 'node[type = "other"]', "style": {"background-color": "gray", "shape": "polygon"}},

    # classes for match vs unique (can combine with type)
    {"selector": "node.match", "style": {"opacity": 0.7, "border-width": .5, "border-color": "#444"}},
    {"selector": "node.unique", "style": {"opacity": 1.0, "border-width": 3, "border-color": "#b30000"}},

    {"selector": "edge.match", "style": {"line-color": "#92adda", "target-arrow-color": "#92adda", "opacity": 0.7}},
    {"selector": "edge.unique", "style": {"line-color": "#ff6b6b", "target-arrow-color": "#ff6b6b", "line-style": "dashed", "opacity": 1.0}},
]

app = Dash(__name__)

app.layout = html.Div([
    html.Div([
        html.H3(file1.name, style={"textAlign": "center", "margin": "6px"}),
        cyto.Cytoscape(
            id="cytoscape-1",
            elements=elements1,
            stylesheet=stylesheet,
            style={"width": "48vw", "height": "80vh", "display": "inline-block"},
            layout={"name": "preset"}
        )
    ], style={"display": "inline-block", "verticalAlign": "top", "width": "49%"}),

    html.Div([
        html.H3(file2.name, style={"textAlign": "center", "margin": "6px"}),
        cyto.Cytoscape(
            id="cytoscape-2",
            elements=elements2,
            stylesheet=stylesheet,
            style={"width": "48vw", "height": "80vh", "display": "inline-block"},
            layout={"name": "preset"}
        )
    ], style={"display": "inline-block", "verticalAlign": "top", "width": "49%"}),
], style={"display": "flex", "justifyContent": "space-between", "padding": "8px"})

if __name__ == "__main__":
    app.run(port=8080, debug=True)